In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
import os
import threading
import tkinter as tk
from tkinter import ttk, scrolledtext, filedialog, messagebox


# ==================== ЛОГИКА ВЕБ-ИНДЕКСАТОРА ====================

def get_links_and_images(url):
    """
    Собирает ссылки и изображения с заданной веб-страницы.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при получении страницы {url}: {e}")
        return [], []

    soup = BeautifulSoup(response.content, 'html.parser')

    links = []
    for link_tag in soup.find_all('a', href=True):
        href = link_tag['href']
        absolute_url = urljoin(url, href)
        links.append(absolute_url)

    images = []
    for img_tag in soup.find_all('img', src=True):
        src = img_tag['src']
        absolute_img_url = urljoin(url, src)
        images.append(absolute_img_url)

    return links, images


def get_extension_from_url(url):
    """
    Извлекает расширение файла из URL, игнорируя якоря (#).
    """
    try:
        url_without_fragment = urldefrag(url)[0]
        parsed_url = urlparse(url_without_fragment)
        path = parsed_url.path
        filename = os.path.basename(path)
        _, ext = os.path.splitext(filename)
        if ext and ext.startswith('.'):
            return ext.lower()
    except Exception as e:
        print(f"Ошибка при извлечении расширения из URL {url}: {e}")
    return None


def download_all_images(image_urls, output_dir="indexed_images"):
    """
    Загружает ВСЕ изображения по URL, определяя расширение из ссылки.
    """
    os.makedirs(output_dir, exist_ok=True)
    downloaded_count = 0
    image_extensions = ['.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp',
                        '.svg', '.ico', '.avif', '.tiff', '.tif', '.heic']

    for img_url in image_urls:
        extension = get_extension_from_url(img_url)

        if extension and extension in image_extensions:
            try:
                headers = {
                    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
                }
                img_response = requests.get(img_url, stream=True, headers=headers)
                img_response.raise_for_status()

                content_type = img_response.headers.get('content-type', '')
                is_valid_image = any(img_type in content_type for img_type in
                                     ['image/', 'application/octet-stream'])

                if is_valid_image or extension == '.svg':
                    filename = os.path.join(output_dir, f"image_{downloaded_count}{extension}")

                    if extension == '.svg':
                        with open(filename, 'w', encoding='utf-8') as f:
                            f.write(img_response.text)
                    else:
                        with open(filename, 'wb') as f:
                            for chunk in img_response.iter_content(chunk_size=8192):
                                f.write(chunk)

                    print(f"✓ Загружено: {img_url} -> {filename}")
                    downloaded_count += 1
                else:
                    print(f"✗ Неверный Content-Type для {img_url}: {content_type}")

            except requests.exceptions.RequestException as e:
                print(f"✗ Ошибка при загрузке {img_url}: {e}")
            except Exception as e:
                print(f"✗ Неизвестная ошибка при обработке {img_url}: {e}")
        else:
            print(f"→ Игнорируем (не изображение): {img_url} (расширение: {extension})")

    print(f"\n✅ Загружено {downloaded_count} изображений в папку '{output_dir}'.")


def save_links(links, output_dir="indexed_data"):
    """
    Сохраняет собранные ссылки в текстовый файл.
    """
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "links.txt"), "w", encoding="utf-8") as f:
        for link in links:
            f.write(link + "\n")
    print(f"✅ Сохранено {len(links)} ссылок в {os.path.join(output_dir, 'links.txt')}")


# ==================== GUI ПРИЛОЖЕНИЕ ====================

class WebIndexerApp:
    """Главное окно приложения веб-индексатора."""

    # Цветовая палитра
    BG_DARK = "#1e1e2e"
    BG_MEDIUM = "#2a2a3d"
    BG_LIGHT = "#363650"
    ACCENT = "#7c3aed"
    ACCENT_HOVER = "#9b5de5"
    TEXT_PRIMARY = "#e0e0e0"
    TEXT_SECONDARY = "#a0a0b8"
    SUCCESS = "#4ade80"
    ERROR = "#f87171"
    WARNING = "#fbbf24"
    BORDER = "#44446a"

    def __init__(self, root):
        self.root = root
        self.root.title("Веб-Индексатор")
        self.root.geometry("900x700")
        self.root.minsize(750, 550)
        self.root.configure(bg=self.BG_DARK)

        # Переменные состояния
        self.all_links = []
        self.all_image_urls = []
        self.is_running = False
        self.output_dir_images = "indexed_images"
        self.output_dir_links = "indexed_data"

        self._setup_styles()
        self._build_ui()

    # ---------- настройка стилей ttk ----------
    def _setup_styles(self):
        self.style = ttk.Style()
        self.style.theme_use("clam")

        self.style.configure("Dark.TFrame", background=self.BG_DARK)
        self.style.configure("Card.TFrame", background=self.BG_MEDIUM)
        self.style.configure(
            "Title.TLabel",
            background=self.BG_DARK,
            foreground=self.ACCENT,
            font=("Segoe UI", 22, "bold"),
        )
        self.style.configure(
            "Subtitle.TLabel",
            background=self.BG_DARK,
            foreground=self.TEXT_SECONDARY,
            font=("Segoe UI", 10),
        )
        self.style.configure(
            "Card.TLabel",
            background=self.BG_MEDIUM,
            foreground=self.TEXT_PRIMARY,
            font=("Segoe UI", 10),
        )
        self.style.configure(
            "CardBold.TLabel",
            background=self.BG_MEDIUM,
            foreground=self.TEXT_PRIMARY,
            font=("Segoe UI", 10, "bold"),
        )
        self.style.configure(
            "Status.TLabel",
            background=self.BG_DARK,
            foreground=self.TEXT_SECONDARY,
            font=("Segoe UI", 9),
        )
        # Прогресс-бар
        self.style.configure(
            "Accent.Horizontal.TProgressbar",
            troughcolor=self.BG_LIGHT,
            background=self.ACCENT,
            thickness=6,
        )

    # ---------- построение интерфейса ----------
    def _build_ui(self):
        # Главный контейнер с отступами
        main = ttk.Frame(self.root, style="Dark.TFrame")
        main.pack(fill="both", expand=True, padx=24, pady=18)

        # --- Заголовок ---
        ttk.Label(main, text="Веб-Индексатор", style="Title.TLabel").pack(anchor="w")
        ttk.Label(
            main,
            text="Сканирование ссылок и загрузка изображений с веб-страницы",
            style="Subtitle.TLabel",
        ).pack(anchor="w", pady=(0, 14))

        # --- Карточка ввода URL ---
        url_card = self._make_card(main)
        url_card.pack(fill="x", pady=(0, 10))

        inner = ttk.Frame(url_card, style="Card.TFrame")
        inner.pack(fill="x", padx=16, pady=14)

        ttk.Label(inner, text="URL веб-сайта:", style="CardBold.TLabel").pack(anchor="w")

        entry_frame = tk.Frame(inner, bg=self.BG_LIGHT, highlightbackground=self.BORDER,
                               highlightthickness=1, bd=0)
        entry_frame.pack(fill="x", pady=(6, 0))

        self.url_entry = tk.Entry(
            entry_frame,
            font=("Consolas", 12),
            bg=self.BG_LIGHT,
            fg=self.TEXT_PRIMARY,
            insertbackground=self.ACCENT,
            relief="flat",
            bd=8,
        )
        self.url_entry.pack(fill="x")
        self.url_entry.insert(0, "https://")
        self.url_entry.bind("<Return>", lambda e: self._start_indexing())

        # --- Кнопки ---
        btn_row = tk.Frame(main, bg=self.BG_DARK)
        btn_row.pack(fill="x", pady=(4, 10))

        self.start_btn = self._make_button(btn_row, "▶  Начать индексацию", self._start_indexing)
        self.start_btn.pack(side="left")

        self.save_links_btn = self._make_button(
            btn_row, "Сохранить ссылки", self._save_links_action, enabled=False
        )
        self.save_links_btn.pack(side="left", padx=(10, 0))

        self.download_btn = self._make_button(
            btn_row, "Скачать изображения", self._download_images_action, enabled=False
        )
        self.download_btn.pack(side="left", padx=(10, 0))

        self.folder_btn = self._make_button(
            btn_row, "Папка сохранения", self._choose_folder
        )
        self.folder_btn.pack(side="right")

        # --- Прогресс ---
        self.progress = ttk.Progressbar(
            main, mode="indeterminate", style="Accent.Horizontal.TProgressbar"
        )
        self.progress.pack(fill="x", pady=(0, 6))

        # --- Статистика ---
        stats_card = self._make_card(main)
        stats_card.pack(fill="x", pady=(0, 10))

        stats_inner = ttk.Frame(stats_card, style="Card.TFrame")
        stats_inner.pack(fill="x", padx=16, pady=10)

        self.stats_links = self._stat_label(stats_inner, "Ссылок:", "0")
        self.stats_links.pack(side="left", padx=(0, 30))

        self.stats_images = self._stat_label(stats_inner, "Изображений:", "0")
        self.stats_images.pack(side="left", padx=(0, 30))

        self.stats_ext = self._stat_label(stats_inner, "Форматы:", "—")
        self.stats_ext.pack(side="left")

        # --- Лог ---
        log_label = ttk.Label(main, text="Журнал операций", style="Subtitle.TLabel")
        log_label.pack(anchor="w", pady=(4, 4))

        log_frame = tk.Frame(main, bg=self.BORDER, bd=1)
        log_frame.pack(fill="both", expand=True)

        self.log_area = scrolledtext.ScrolledText(
            log_frame,
            font=("Consolas", 10),
            bg=self.BG_MEDIUM,
            fg=self.TEXT_PRIMARY,
            insertbackground=self.ACCENT,
            relief="flat",
            wrap="word",
            state="disabled",
            bd=6,
        )
        self.log_area.pack(fill="both", expand=True)

        # Теги для цветного текста
        self.log_area.tag_configure("success", foreground=self.SUCCESS)
        self.log_area.tag_configure("error", foreground=self.ERROR)
        self.log_area.tag_configure("warning", foreground=self.WARNING)
        self.log_area.tag_configure("info", foreground=self.TEXT_SECONDARY)

        # --- Строка состояния ---
        self.status_label = ttk.Label(main, text="Готов к работе", style="Status.TLabel")
        self.status_label.pack(anchor="w", pady=(6, 0))

    # ---------- вспомогательные виджеты ----------
    def _make_card(self, parent):
        """Создаёт «карточку» — рамку с фоном."""
        outer = tk.Frame(parent, bg=self.BORDER, bd=1)
        card = ttk.Frame(outer, style="Card.TFrame")
        card.pack(fill="both", expand=True)
        # Возвращаем outer, чтобы pack/grid применялся к рамке
        card._outer = outer
        return outer

    def _make_button(self, parent, text, command, enabled=True):
        """Создаёт стилизованную кнопку."""
        btn = tk.Button(
            parent,
            text=text,
            font=("Segoe UI", 10, "bold"),
            bg=self.ACCENT,
            fg="white",
            activebackground=self.ACCENT_HOVER,
            activeforeground="white",
            relief="flat",
            cursor="hand2",
            padx=16,
            pady=6,
            command=command,
        )
        if not enabled:
            btn.configure(state="disabled", bg=self.BG_LIGHT, fg=self.TEXT_SECONDARY)

        # Эффект наведения
        def on_enter(e):
            if btn["state"] != "disabled":
                btn.configure(bg=self.ACCENT_HOVER)

        def on_leave(e):
            if btn["state"] != "disabled":
                btn.configure(bg=self.ACCENT)

        btn.bind("<Enter>", on_enter)
        btn.bind("<Leave>", on_leave)
        return btn

    def _enable_button(self, btn):
        btn.configure(state="normal", bg=self.ACCENT, fg="white")

    def _disable_button(self, btn):
        btn.configure(state="disabled", bg=self.BG_LIGHT, fg=self.TEXT_SECONDARY)

    def _stat_label(self, parent, title, value):
        """Пара «заголовок — значение» для статистики."""
        frame = ttk.Frame(parent, style="Card.TFrame")
        ttk.Label(frame, text=title, style="Card.TLabel").pack(side="left")
        val = ttk.Label(frame, text=value, style="CardBold.TLabel")
        val.pack(side="left", padx=(4, 0))
        frame._value_label = val  # сохраняем ссылку для обновления
        return frame

    def _update_stat(self, stat_frame, value):
        stat_frame._value_label.configure(text=str(value))

    # ---------- логирование ----------
    def _log(self, message, tag="info"):
        """Добавляет строку в журнал."""
        self.log_area.configure(state="normal")
        self.log_area.insert("end", message + "\n", tag)
        self.log_area.see("end")
        self.log_area.configure(state="disabled")

    def _set_status(self, text):
        self.status_label.configure(text=text)

    # ---------- действия ----------
    def _choose_folder(self):
        folder = filedialog.askdirectory(title="Выберите папку для сохранения")
        if folder:
            self.output_dir_images = os.path.join(folder, "indexed_images")
            self.output_dir_links = os.path.join(folder, "indexed_data")
            self._log(f"Папка сохранения: {folder}", "info")

    def _start_indexing(self):
        url = self.url_entry.get().strip()
        if not url or url == "https://":
            messagebox.showwarning("Внимание", "Введите URL для индексации!")
            return
        if self.is_running:
            return

        # Сброс
        self.all_links.clear()
        self.all_image_urls.clear()
        self._disable_button(self.save_links_btn)
        self._disable_button(self.download_btn)
        self._disable_button(self.start_btn)
        self._update_stat(self.stats_links, "0")
        self._update_stat(self.stats_images, "0")
        self._update_stat(self.stats_ext, "—")

        self.log_area.configure(state="normal")
        self.log_area.delete("1.0", "end")
        self.log_area.configure(state="disabled")

        self.is_running = True
        self.progress.start(12)
        self._set_status("Идёт индексация…")
        self._log(f"Индексируем: {url}", "info")

        # Запуск в отдельном потоке, чтобы GUI не зависал
        thread = threading.Thread(target=self._indexing_worker, args=(url,), daemon=True)
        thread.start()

    def _indexing_worker(self, url):
        """Выполняется в фоновом потоке."""
        links, images = get_links_and_images(url)
        # Возвращаемся в главный поток для обновления GUI
        self.root.after(0, self._indexing_done, links, images)

    def _indexing_done(self, links, images):
        """Вызывается в главном потоке после завершения индексации."""
        self.progress.stop()
        self.is_running = False
        self._enable_button(self.start_btn)

        self.all_links = links
        self.all_image_urls = images

        # Обновляем статистику
        self._update_stat(self.stats_links, len(links))
        self._update_stat(self.stats_images, len(images))

        # Расширения
        ext_count = {}
        for u in images:
            ext = get_extension_from_url(u)
            if ext:
                ext_count[ext] = ext_count.get(ext, 0) + 1
        if ext_count:
            ext_str = ", ".join(f"{e}({c})" for e, c in ext_count.items())
            self._update_stat(self.stats_ext, ext_str)

        # Лог
        if links:
            self._log(f"Найдено ссылок: {len(links)}", "success")
            self._enable_button(self.save_links_btn)
        else:
            self._log("Ссылки не найдены.", "warning")

        if images:
            self._log(f"Найдено изображений: {len(images)}", "success")
            for ext, count in ext_count.items():
                self._log(f"   {ext}: {count}", "info")
            self._enable_button(self.download_btn)
        else:
            self._log("Изображения не найдены.", "warning")

        self._set_status("Индексация завершена")

    def _save_links_action(self):
        if not self.all_links:
            return
        self._disable_button(self.save_links_btn)
        self._set_status("Сохраняем ссылки…")

        def worker():
            save_links(self.all_links, self.output_dir_links)
            self.root.after(0, done)

        def done():
            path = os.path.join(self.output_dir_links, "links.txt")
            self._log(f"Ссылки сохранены → {path}", "success")
            self._set_status("Ссылки сохранены")
            self._enable_button(self.save_links_btn)

        threading.Thread(target=worker, daemon=True).start()

    def _download_images_action(self):
        if not self.all_image_urls:
            return
        self._disable_button(self.download_btn)
        self.progress.start(12)
        self._set_status("Загружаем изображения…")
        self._log("Начинаем загрузку изображений…", "info")

        def worker():
            download_all_images(self.all_image_urls, self.output_dir_images)
            self.root.after(0, done)

        def done():
            self.progress.stop()
            self._log(f"Изображения сохранены → {self.output_dir_images}", "success")
            self._set_status("Загрузка завершена")
            self._enable_button(self.download_btn)

        threading.Thread(target=worker, daemon=True).start()


# ==================== ЗАПУСК ====================

if __name__ == "__main__":
    root = tk.Tk()
    app = WebIndexerApp(root)
    root.mainloop()
